# Part 1 - Data Warehouse Modeling: Star Schema

## 1.1 Define the Business Process and the Fact Grain

The Business process: Financial transactions of a trading activity recorded in the 2024 account statement.

The Grain of the Fact table: one row in "Fact_Transactions" represents one transaction from the account statement file. We can consider this an event transaction fact, one fact row per business event.


## 1.2 Identify Fact and Dimensions

We have identified the following tables:

- **Fact Table**: Fact_Transactions
  
- **Dimension Tables**:
  - Dim_Time
  - Dim_Geography
  - Dim_Symbol
  - Dim_TransactionType



In **Fact_Transactions** I put **Unit***, which is the measure of the fact, and **IDTransaction**, ia a descriptive attribute of the single transaction.
The other attributes of the account statemnet decsribe the fact becoming dimensions, **Date** tells when the transaction happended and goes into **Dim_Time**; **TransactionType** tells the type of events like Buy or Sell and goes into **Dim_TransactionType**;
**Symbol** tell about what the traded and goes into **Dim_Symbol**, togheter with ***company_name, industry and sector*** that describe the traded stock, comes from the symbol file.

The attribute **country** is the difficult one, becouse come from the 'symbol file' and in a first moment seems as an attribute of **Dim_Symbol**, but the assignment asks for a separate **Dim_Geography** and tells us to avoid unnecessary duplication trough dimensions, to this specific reason I decided to keep **country** only in **Dim_Geography**, togheter with ***region and sub-region*** from the 'country' file.

The symbols file is used only during the loading phase, to connect every transaction to the country of its company. Also I did not include in the model the columns of the country file that are not useful for any analysis


Attribute:

- Fact_Transaction: --> IDTransaction, Unit
- Dim_time: --> Date
- Dim_TransactionType: --> TransactionType
- Dim_Symbol: --> Symbol, company_name, sector, industry
- Dim_Geography: --> country, region, sub-region

## 1.3 Dimension Hierarchies

- Dim_Time: Day -> Month -> Quarter -> Year
- Dim_Geography: Country -> Region -> Sub-region
- Dim_Symbol: Symbol -> Industry -> Sector
- Dim_TransactionType: No hierarchy 


In Dim_Symbol, each simboly come from one industry, and each industry has just one sector, the the hierarchies is symbol, industry and sector

in Dim_TransactionType, we have no levels just an attribute with 2 value, so no hierarchy

## 1.4 Final Star Schema

**Fact_Transactions (time_sk, symbol_sk, geo_sk, type_sk, id_transaction, units)** contains 4 foreign key that aims inthe same dimensions

**Dim_Time (time_sk, date, day, month, quarter, year)** time_sk is the primary key and date is teh bottom

**Dim_Symbol (symbol_sk, symbol, company_name, industry, sector)**

**Dim_Geography (geo_sk, country, region, sub_region)**

**Dim_TransactionType (type_sk, transaction_type)** #just one attribute



Each dimension tablse has a surrogate primary key
- time_sk
- symbol_sk
- geo_sk
- type_sk

The fact table references all the dimensions tables thourght the foreign keys, instead the primary key is the composisione of the foreign key.
**units** is the measure of the fact aggregade with the SUM operation;
**id_transaction** is a descriptive attribute.
The number of of transactions is compued by couting the fact rows

# Part 2 - Data Transformation and Analysis


In [1]:
import pandas as pd

# to display more columns when we inspect the dataset
pd.set_option("display.max_columns", 100)

tx_raw = pd.read_csv("Datasets/account-statement-1-1-2024-12-31-2024.csv", sep=';', encoding = 'utf-8-sig')


symbols = pd.read_csv('Datasets/symbols.csv', sep=';', encoding = 'utf-8-sig')

country = pd.read_csv('Datasets/country.csv')

print("Account statement shape:", tx_raw.shape)
print("Symbols shape:", symbols.shape)
print("Country shape:", country.shape)

tx_raw.head()

Account statement shape: (2745, 6)
Symbols shape: (3194, 5)
Country shape: (249, 11)


,IDTransaction,Date,TransactionType,Symbol,Unit,Unnamed: 5
0,2.769834e+09,11/01/2024 10:44:03,BUY,BAP,1605.0,NaN
1,2.767325e+09,24/01/2024 08:07:24,SELL,BAP,1605.0,NaN
2,2.815474e+09,10/01/2024 11:00:08,SELL,BAP,914.0,NaN
3,2.622244e+09,16/01/2024 08:14:21,BUY,ACGL,646.0,NaN
4,2.629871e+09,16/01/2024 14:34:12,SELL,ALVO,646.0,NaN


In [2]:
# check ispect columns and missing values

summary = pd.DataFrame({
    "column": tx_raw.columns,
    "dtype": tx_raw.dtypes.astype(str).values,
    "missing_values": tx_raw.isna().sum().values,
    "missing_pct": (tx_raw.isna().mean().values * 100).round(2)
})

summary

,column,dtype,missing_values,missing_pct
0,IDTransaction,float64,464,16.9
1,Date,str,464,16.9
2,TransactionType,str,464,16.9
3,Symbol,str,464,16.9
4,Unit,float64,464,16.9
5,Unnamed: 5,float64,2745,100.0


We can noticed an empty extra column every line of the file ends with ";" and a group of completely blank rows. 

In [3]:
# 4. Clean the transaction log


# We start from the raw dataset
tx = tx_raw.copy()

# Remove the empty extra column created by ";" and blanck rows
tx = tx.drop(columns=[c for c in tx.columns if c.startswith("Unnamed")])


tx = tx.dropna(how="all")

# Convert the columns to the correct types, so Dates are in day/month/year format

tx["IDTransaction"] = tx["IDTransaction"].astype("int64")
tx["Unit"] = tx["Unit"].astype("int64")
tx["Date"] = pd.to_datetime(tx["Date"], format="%d/%m/%Y %H:%M:%S")

print("Valid transactions:", len(tx))
print("Transaction types:", tx["TransactionType"].value_counts().to_dict())

Valid transactions: 2281
Transaction types: {'SELL': 1089, 'BUY': 1082, 'DIVIDENT': 110}


In [4]:


# 5. Verify that every transaction symbol exists in the symbol dataset


# transactions whose symbol is not in symbols.csv cannot be described
# by Dim_Symbol and Dim_Geography, so they must be identified
unknown = ~tx["Symbol"].isin(symbols["symbol"])

print("Transactions with unknown symbol:", unknown.sum(), "of", len(tx))
print("Symbols not found:", sorted(tx.loc[unknown, "Symbol"].unique()))

# We remove them, because they cannot be linked to the dimensions
tx = tx[~unknown].copy()
print("Transactions kept:", len(tx))

Transactions with unknown symbol: 212 of 2281
Symbols not found: ['AGO.l', 'ARCH', 'AZM', 'CCAP', 'CSIQ', 'FNC', 'HTGC', 'IBE', 'MFG', 'MONC', 'OBDC', 'RCMT', 'RIGZU', 'SAP', 'TKC', 'UCG', 'VWS', 'WF']
Transactions kept: 2069


In [5]:
# 6. Verify that every company country can be mapped to the country dataset


# Two country names do not match the NAME_FIXES used in country.csv,
# so we fix them before the merge
NAME_FIXES = {"Turkey": "Türkiye", "Taiwan": "Taiwan, Province of China"}
symbols["country"] = symbols["country"].replace(NAME_FIXES)

# After the fix, every company country must be mappable.
not_mapped = set(symbols["country"]) - set(country["name"])
print("Countries not mappable:", not_mapped or "none")

Countries not mappable: none


**Create the dataframes required by the star schema**

Each dataframe keeps only the information included in the dimensional model.

In [6]:
# CREATION OF THE STAR SCHEMA


# --- Dim_Time: Day -> Month -> Quarter -> Year ---
dim_time = (tx['Date'].dt.normalize().drop_duplicates().sort_values()
            .rename('date').to_frame().reset_index(drop=True))
dim_time['day']     = dim_time['date'].dt.day
dim_time['month']   = dim_time['date'].dt.month
dim_time['quarter'] = dim_time['date'].dt.quarter
dim_time['year']    = dim_time['date'].dt.year
dim_time.insert(0, 'time_sk', dim_time.index + 1)

# --- Dim_Symbol: Symbol -> Industry -> Sector ---
dim_symbol = (symbols[['symbol', 'company_name', 'industry', 'sector']]
              .drop_duplicates('symbol').reset_index(drop=True))
dim_symbol.insert(0, 'symbol_sk', dim_symbol.index + 1)

# --- Dim_Geography: Country -> Region -> Sub-region ---
dim_geography = (symbols[['country']].drop_duplicates()
                 .merge(country[['name', 'region', 'sub-region']],
                        left_on='country', right_on='name', how='left')
                 [['country', 'region', 'sub-region']]
                 .rename(columns={'sub-region': 'sub_region'})
                 .sort_values('country').reset_index(drop=True))
dim_geography.insert(0, 'geo_sk', dim_geography.index + 1)

# --- Dim_TransactionType ---
dim_transactiontype = (tx[['TransactionType']].drop_duplicates()
                       .rename(columns={'TransactionType': 'transaction_type'})
                       .sort_values('transaction_type').reset_index(drop=True))
dim_transactiontype.insert(0, 'type_sk', dim_transactiontype.index + 1)

for name, d in [('Dim_Time', dim_time), ('Dim_Symbol', dim_symbol),
                ('Dim_Geography', dim_geography), ('Dim_TransactionType', dim_transactiontype)]:
    print(f'{name}: {len(d)} rows, columns {list(d.columns)}')

Dim_Time: 228 rows, columns ['time_sk', 'date', 'day', 'month', 'quarter', 'year']
Dim_Symbol: 3194 rows, columns ['symbol_sk', 'symbol', 'company_name', 'industry', 'sector']
Dim_Geography: 42 rows, columns ['geo_sk', 'country', 'region', 'sub_region']
Dim_TransactionType: 3 rows, columns ['type_sk', 'transaction_type']


In [7]:
fact = (tx
    .assign(date=tx['Date'].dt.normalize())
    .merge(dim_time[['date', 'time_sk']], on='date')
    .merge(dim_symbol[['symbol', 'symbol_sk']], left_on='Symbol', right_on='symbol')
    .merge(symbols[['symbol', 'country']], on='symbol')          # ETL lookup symbol -> country
    .merge(dim_geography[['country', 'geo_sk']], on='country')
    .merge(dim_transactiontype, left_on='TransactionType', right_on='transaction_type')
    [['IDTransaction', 'time_sk', 'symbol_sk', 'geo_sk', 'type_sk', 'Unit']]
    .rename(columns={'IDTransaction': 'id_transaction', 'Unit': 'units'})
    .reset_index(drop=True))
fact.insert(0, 'transaction_sk', fact.index + 1)

# integrity checks
assert len(fact) == len(tx), 'row count changed during FK resolution'
assert fact[['time_sk', 'symbol_sk', 'geo_sk', 'type_sk']].notna().all().all(), 'unresolved FK'
assert fact['transaction_sk'].is_unique
print(f'Fact_Transactions: {len(fact)} rows — all FKs resolved')
fact.head()

Fact_Transactions: 2069 rows — all FKs resolved


,transaction_sk,id_transaction,time_sk,symbol_sk,geo_sk,type_sk,units
0,1,2769834124,8,284,32,1,1605
1,2,2767324642,17,284,32,3,1605
2,3,2815473914,7,284,32,3,914
3,4,2622244212,11,4,3,1,646
4,5,2629871124,11,258,26,3,646


**Building the fact table**

The symbol's country is used here *once*, as an ETL-time lookup, to assign `geo_sk` to each fact row — it is not stored in `Dim_Symbol` (no attribute duplication across dimensions). `IDTransaction` is not unique in the source, so the primary key is the generated surrogate `transaction_sk`.

## 2.2 Analytical Questions



Five questions are answered: **3, 4, 5, 7, 11**. Together they exercise every dimension of the schema: time (Q3, Q11), geography at country / region / sub-region level (Q4, Q5, Q11), symbol (Q7) and transaction type (Q4, Q5).

All queries follow the OLAP pattern seen in class: join the fact table with the needed dimension tables, filter, group by the dimension attributes, aggregate (`COUNT(*)` for number of transactions, `SUM(units)` for traded units).

In [8]:
# One reusable "star join" of the fact with its dimensions
star = (fact
        .merge(dim_time, on='time_sk')
        .merge(dim_symbol, on='symbol_sk')
        .merge(dim_geography, on='geo_sk')
        .merge(dim_transactiontype, on='type_sk'))
buy_sell = star[star['transaction_type'].isin(['BUY', 'SELL'])]
print(f'star view: {len(star)} rows, of which BUY+SELL: {len(buy_sell)}')

star view: 2069 rows, of which BUY+SELL: 1973


**Q3 — Rank all quarters of 2024 by total number of transactions (BUY + SELL).**

In [9]:
# Group by quarter and count the fact rows


q3 = (buy_sell[buy_sell['year'] == 2024]
      .groupby('quarter')
      .size()
      .sort_values(ascending=False)
      .rename('n_transactions')
      .reset_index())
q3

,quarter,n_transactions
0,1,968
1,2,522
2,3,242
3,4,241


I filtered trading transictions only BUY/SELL 2024, group by Quarter (Dim_Time) and count the rows.
We can analysi a strong concentration in Q1 and a declining in the next few quarters

**Q4 — What are the top 10 countries by number of SELL transactions in 2024?.**

In [10]:
q4 = (star[(star['transaction_type'] == 'SELL') & (star['year'] == 2024)]
      .groupby('country')
      .size()
      .nlargest(10).rename('n_sell_transactions')
      .reset_index())
q4

,country,n_sell_transactions
0,United States of America,389
1,United Kingdom of Great Britain and Northern I...,130
2,China,112
3,Brazil,69
4,"Taiwan, Province of China",50
5,"Netherlands, Kingdom of the",46
6,Switzerland,37
7,Ireland,31
8,Luxembourg,27
9,Canada,22


I filtered Transaction_type 0 sell and Year = 2024, I didi the Group by country (Dim_Geography) counting the transactions.
USA leading the trasnsaction by 389 with the following Uk and China

**Q5 — What are the top 5 regions by total units bought in 2024?**

*Note: only three regions (Americas, Europe, Asia) appear among the BUY transactions, so the "top 5" contains three rows.*

In [11]:
q5 = (star[(star['transaction_type'] == 'BUY') & (star['year'] == 2024)]
      .groupby('region')['units'].sum()
      .nlargest(5).rename('units_bought')
      .reset_index())
q5

,region,units_bought
0,Americas,37026
1,Europe,22528
2,Asia,9198


I filtered BUY transaction in 2024 and group by region, and as teh measure sum of units insted counting it
I select then unit column and apply sum instead of using .size. Becouse the Q5 asks fot "total units bought", not "number of transactions"

**Q7 — What are the top 10 symbols by number of transactions (BUY + SELL) in 2024?**

In [12]:
q7 = (buy_sell[buy_sell['year'] == 2024]
      .groupby('symbol').size()
      .nlargest(10).rename('n_transactions')
      .reset_index())
q7

,symbol,n_transactions
0,ARM,100
1,AMD,97
2,TSM,77
3,TIMB,76
4,GOOG,48
5,AMZN,47
6,MSFT,45
7,ARDX,43
8,BRFS,42
9,EH,40


I filtered Year and group by simbol ( Dim_Symbol) and count the rows as I did for Q3.
We'll see the same symbols in the streamlit dashbord in the chart 3, as the top 3 traded symbols

**Q11 — What are the top 5 sub-regions by number of transactions in Q4 of 2024?**

In [13]:
q11 = (buy_sell[(buy_sell['year'] == 2024) & (buy_sell['quarter'] == 4)]
       .groupby('sub_region').size()
       .nlargest(5).rename('n_transactions')
       .reset_index())
q11

,sub_region,n_transactions
0,Northern America,124
1,Eastern Asia,28
2,Northern Europe,28
3,Latin America and the Caribbean,23
4,Western Europe,23


I filtered Q4 of 2024 and group by Subregion. I use .size for transaction count

## Part 3 — Streamlit Dashboard

The dashboard (first exam session: **Page 1 — Time Analysis** only) is delivered separately in `streamlit_app.zip`. It reads the star schema exported above (`streamlit_app/data/`) and provides:

- a date-range filter (defaults 01/01/2024 – 31/12/2024) that drives every chart;
- a line chart of total BUY+SELL transactions over time;
- bar charts with the top 3 symbols, top 5 sectors and top 5 industries by transaction count.

To run it: `pip install -r requirements.txt` then

`streamlit run Home.py` inside the app folder.

In [14]:
# Export the star schema tables used by the Streamlit dashboard
import os
os.makedirs("streamlit_app/data", exist_ok=True)

dim_time.to_csv("streamlit_app/data/dim_time.csv", index=False)
dim_geography.to_csv("streamlit_app/data/dim_geography.csv", index=False)
dim_symbol.to_csv("streamlit_app/data/dim_symbol.csv", index=False)
dim_transactiontype.to_csv("streamlit_app/data/dim_transactiontype.csv", index=False)
fact.to_csv("streamlit_app/data/fact_transactions.csv", index=False)

print("5 CSV files exported to streamlit_app/data/")

5 CSV files exported to streamlit_app/data/
